In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "spectral.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

In [2]:
line_mesh(n=3, order=3)

mat = Material("body")

P = Problem(
    [mat];
    type=:ScalarField,
    dim=1,
    fieldName=:u,
    basis=:spectral
)

support = BoundaryCondition("left", u=1)
K = ∫(
    Grad(P) ⋅ Grad(P);
    Ω="body"
)

f = ∫(
    P ⋅ 1;
    Ω="body"
)

u = solveField(
    Symmetric(K),
    f;
    support=[support]
)

nodal ScalarField
[1.0; 1.4999999999999858; … ; 1.4753086419752959; 1.4938271604938131;;]

In [3]:
showDoFResults(u, name="u spectral")

0

In [4]:
#line_mesh(n=3, order=3)

#mat = Material("body")

P2 = Problem(
    [mat];
    type=:ScalarField,
    dim=1,
    fieldName=:u,
    basis=:lagrange
)

support = BoundaryCondition("left", u=1)
K = ∫(
    Grad(P2) ⋅ Grad(P2);
    Ω="body"
)

f = ∫(
    P2 ⋅ 1;
    Ω="body"
)

u2 = solveField(
    Symmetric(K),
    f;
    support=[support]
)

nodal ScalarField
[1.0; 1.4999999999999944; … ; 1.4753086419753034; 1.4938271604938214;;]

In [5]:
showDoFResults(u2, name="u lagrange")

1

In [6]:
norm(DoFs(u) - DoFs(u2)) / norm(DoFs(u))

3.778032099261138e-15

In [7]:
Ts, Rs = LowLevelFEM.field_transformation(P)
Tl, Rl = LowLevelFEM.field_transformation(P2)

@show P.basis
@show P2.basis

@show size(Ts)
@show size(Tl)

@show nnz(Ts)
@show nnz(Tl)

@show norm(Ts - Tl, Inf)

P.basis = :spectral
P2.basis = :lagrange
size(Ts) = (10, 10)
size(Tl) = (10, 10)
nnz(Ts) = 28
nnz(Tl) = 10
norm(Ts - Tl, Inf) = 1.0740740740740742


1.0740740740740742

In [8]:
Tl

10×10 SparseMatrixCSC{Float64, Int64} with 10 stored entries:
 1.0   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅   1.0   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅   1.0   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅   1.0   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅   1.0   ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅   1.0   ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   1.0   ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   1.0   ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   1.0   ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   1.0

In [9]:
Ts

10×10 SparseMatrixCSC{Float64, Int64} with 28 stored entries:
  1.0         ⋅         ⋅        …   ⋅         ⋅          ⋅ 
   ⋅          ⋅         ⋅            ⋅         ⋅         1.0
   ⋅          ⋅         ⋅            ⋅         ⋅          ⋅ 
   ⋅          ⋅         ⋅            ⋅         ⋅          ⋅ 
 -0.0740741  0.969642  0.141469      ⋅         ⋅          ⋅ 
 -0.037037   0.141469  0.969642  …   ⋅         ⋅          ⋅ 
   ⋅          ⋅         ⋅            ⋅         ⋅          ⋅ 
   ⋅          ⋅         ⋅            ⋅         ⋅          ⋅ 
   ⋅          ⋅         ⋅           0.969642  0.141469  -0.037037
   ⋅          ⋅         ⋅           0.141469  0.969642  -0.0740741

In [10]:
#openPostProcessor()